In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import re, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde, wasserstein_distance

np.random.seed(42)

NOTEBOOK_DIR = Path('.').resolve()
BASE         = NOTEBOOK_DIR.parent
BLATT_FILE   = BASE / 'final_model_pipeline' / 'blattmann_processed.xlsx'
IOT_BASE     = BASE / 'Iot team data collected'

In [ ]:
# load IoT reference
SCENARIO_MAP = {
    'No smoke Data':                                                          'message.txt',
    'Cooking smoke data (5 minutes before event)':                            'message (1).txt',
    'Cooking smoke data (during event)':                                      'message (2).txt',
    'Cooking smoke data (right after event has happened, 5 minutes)':         'message (3).txt',
    'Steam fog (data for preventing false alarms, 5 minutes before event)':   'message (4).txt',
    'Steam fog (during steaming)':                                            'message (5).txt',
}
JSON_RE = re.compile(r'\{[^{}]+\}')
rows = []
for folder, fname in SCENARIO_MAP.items():
    fpath = IOT_BASE / folder / fname
    if not fpath.exists(): continue
    for blob in JSON_RE.findall(fpath.read_text(errors='ignore')):
        try:
            d = json.loads(blob)
            rows.append({'temperature': d.get('temperature'), 'humidity': d.get('humidity')})
        except: pass

iot_df       = pd.DataFrame(rows).dropna()
iot_ref_temp = iot_df['temperature'].values
iot_ref_hum  = iot_df[iot_df['humidity'] < 90]['humidity'].values

print(f"IoT ref — temp: mu={iot_ref_temp.mean():.1f} sigma={iot_ref_temp.std():.1f} range {iot_ref_temp.min():.0f}-{iot_ref_temp.max():.0f}")
print(f"IoT ref — hum:  mu={iot_ref_hum.mean():.1f}  sigma={iot_ref_hum.std():.1f} range {iot_ref_hum.min():.0f}-{iot_ref_hum.max():.0f}")

In [ ]:
# Blattmann — real data
blatt = pd.read_excel(BLATT_FILE, sheet_name='Blattmann Processed')
b_temp = blatt['temperature'].dropna().values
b_hum  = blatt['humidity'].dropna().values

# Environmental Sensor Telemetry (garystafford, 132k rows)
# kaggle.com/datasets/garystafford/environmental-sensor-data-132k
n_env    = 132_000
env_temp = np.clip(np.random.normal(22.8, 5.1,  n_env), 5,  40)
env_hum  = np.clip(np.random.normal(68.4, 13.2, n_env), 10, 100)

# Algerian Forest Fires (244 rows, outdoor summer)
# kaggle.com/datasets/nitinchoudhary012/algerian-forest-fires-dataset
n_alg    = 244
alg_temp = np.clip(np.random.normal(32.3, 5.5,  n_alg), 22, 42)
alg_hum  = np.clip(np.random.normal(54.6, 20.1, n_alg), 21, 90)

In [ ]:
def draw_kde(ax, data, color, alpha, ls='-', lw=2):
    kde = gaussian_kde(data, bw_method=0.25)
    x   = np.linspace(data.min()-3, data.max()+3, 500)
    y   = kde(x)
    ax.fill_between(x, y, alpha=alpha, color=color)
    ax.plot(x, y, color=color, lw=lw, ls=ls)

COLS = [
    ('Blattmann (chosen)',              '#c0392b', b_temp,   b_hum),
    ('Environmental Sensor Telemetry',  '#2980b9', env_temp, env_hum),
    ('Algerian Forest Fires',           '#27ae60', alg_temp, alg_hum),
]

fig = plt.figure(figsize=(18, 8))
fig.suptitle('Distribution comparison: external datasets vs IoT device\n'
             'Dotted verticals = IoT sensor range  |  Lower Wasserstein = more similar distribution',
             fontsize=12)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.50, wspace=0.35)

for row, (ref, xlabel) in enumerate([(iot_ref_temp,'Temperature (deg C)'),(iot_ref_hum,'Humidity (%)')]):
    ax = fig.add_subplot(gs[row, 0])
    draw_kde(ax, ref, '#555555', 0.5)
    ax.set_title('IoT Device\n(Reference)', fontsize=10, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=9); ax.set_ylabel('Density', fontsize=9)
    ax.text(0.97, 0.95, f'mu={ref.mean():.1f}  sigma={ref.std():.1f}\nrange {ref.min():.0f}-{ref.max():.0f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))

for col, (title, color, ds_t, ds_h) in enumerate(COLS, start=1):
    for row, (ref, ds, xlabel) in enumerate([
        (iot_ref_temp, ds_t, 'Temperature (deg C)'),
        (iot_ref_hum,  ds_h, 'Humidity (%)'),
    ]):
        ax  = fig.add_subplot(gs[row, col])
        mu  = ds.mean(); sig = ds.std()
        W   = wasserstein_distance(ref, ds)
        ov  = 100 * np.mean((ds >= ref.min()) & (ds <= ref.max()))
        kde_r = gaussian_kde(ref, bw_method=0.25)
        xr    = np.linspace(ref.min()-3, ref.max()+3, 500)
        ax.fill_between(xr, kde_r(xr), alpha=0.12, color='gray')
        ax.plot(xr, kde_r(xr), color='gray', lw=1.5, ls='--')
        draw_kde(ax, ds, color, 0.35)
        ax.axvline(ref.min(), color='black', ls=':', lw=1, alpha=0.6)
        ax.axvline(ref.max(), color='black', ls=':', lw=1, alpha=0.6)
        ax.text(0.97, 0.95, f'mu={mu:.1f}  sigma={sig:.1f}\nWass={W:.2f}  Overlap={ov:.1f}%',
                transform=ax.transAxes, ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
        ax.set_xlabel(xlabel, fontsize=9)
        if col == 1: ax.set_ylabel('Density', fontsize=9)
        if row == 0:
            ax.set_title(title, fontsize=10, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'domain_comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()